## Create a dataframe called views with two columns: datetime and user by reading feed-views.log

In [1]:
import pandas as pd
views = pd.read_csv('../data/feed-views.log',
                 names = ['datetime', 'user'],
                 sep = '\t',
                 engine = 'python',
                 skiprows=[2,3],          # Пропускаем строки 2 и 3
                 skipfooter=2)
views['datetime'] = views['datetime'].astype("datetime64[ns]")
views['year'] = views['datetime'].dt.year
views['month'] = views['datetime'].dt.month
views['day'] = views['datetime'].dt.day
views['hour'] = views['datetime'].dt.hour
views['minute'] = views['datetime'].dt.minute
views['second'] = views['datetime'].dt.second
views

,datetime,user,year,month,day,hour,minute,second
0,2020-04-17 12:01:08.463179,artem,2020,4,17,12,1,8
1,2020-04-17 12:01:23.743946,artem,2020,4,17,12,1,23
2,2020-04-17 12:35:52.735016,artem,2020,4,17,12,35,52
3,2020-04-17 12:36:21.401412,oksana,2020,4,17,12,36,21
4,2020-04-17 12:36:22.023355,oksana,2020,4,17,12,36,22
...,...,...,...,...,...,...,...,...
1067,2020-05-21 16:36:40.915488,ekaterina,2020,5,21,16,36,40
1068,2020-05-21 17:49:36.429237,maxim,2020,5,21,17,49,36
1069,2020-05-21 18:45:20.441142,valentina,2020,5,21,18,45,20
1070,2020-05-21 23:03:06.457819,maxim,2020,5,21,23,3,6


## Create the new column daytime.

In [2]:
views.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1072 entries, 0 to 1071
Data columns (total 8 columns):
 #   Column    Non-Null Count  Dtype         
---  ------    --------------  -----         
 0   datetime  1072 non-null   datetime64[ns]
 1   user      1072 non-null   object        
 2   year      1072 non-null   int32         
 3   month     1072 non-null   int32         
 4   day       1072 non-null   int32         
 5   hour      1072 non-null   int32         
 6   minute    1072 non-null   int32         
 7   second    1072 non-null   int32         
dtypes: datetime64[ns](1), int32(6), object(1)
memory usage: 42.0+ KB


In [3]:
cut_labels = ['night', 'early morning', 'morning', 'afternoon', 'early evening', 'evening']
cut_bins = [0, 4, 7, 11, 17, 20, 24]

views['daytime'] = pd.cut(views['hour'],
                       bins=cut_bins,
                       labels=cut_labels,
                       right=False)
views = views.set_index('user')
views

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
artem,2020-04-17 12:01:08.463179,2020,4,17,12,1,8,afternoon
artem,2020-04-17 12:01:23.743946,2020,4,17,12,1,23,afternoon
artem,2020-04-17 12:35:52.735016,2020,4,17,12,35,52,afternoon
oksana,2020-04-17 12:36:21.401412,2020,4,17,12,36,21,afternoon
oksana,2020-04-17 12:36:22.023355,2020,4,17,12,36,22,afternoon
...,...,...,...,...,...,...,...,...
ekaterina,2020-05-21 16:36:40.915488,2020,5,21,16,36,40,afternoon
maxim,2020-05-21 17:49:36.429237,2020,5,21,17,49,36,early evening
valentina,2020-05-21 18:45:20.441142,2020,5,21,18,45,20,early evening


In [4]:
views.count()

datetime    1072
year        1072
month       1072
day         1072
hour        1072
minute      1072
second      1072
daytime     1072
dtype: int64

In [5]:
views.loc[views.daytime == 'night'].hour.idxmax()

'konstantin'

In [6]:
views.loc[views.daytime == 'morning'].hour.idxmin()

'alexander'

## Calculate the number of elements in your dataframe.

In [7]:
views.hour.mode()

0    22
Name: hour, dtype: int32

In [8]:
views.daytime.mode()

0    evening
Name: daytime, dtype: category
Categories (6, object): ['night' < 'early morning' < 'morning' < 'afternoon' < 'early evening' < 'evening']

In [9]:
views.daytime.value_counts()

daytime
evening          508
afternoon        250
early evening    145
night            129
morning           35
early morning      5
Name: count, dtype: int64

In [10]:
print(views.count())
views['daytime'].value_counts()

datetime    1072
year        1072
month       1072
day         1072
hour        1072
minute      1072
second      1072
daytime     1072
dtype: int64


daytime
evening          508
afternoon        250
early evening    145
night            129
morning           35
early morning      5
Name: count, dtype: int64

## Sort the values in your dataframe by hour, minute, and second in ascending order simultaneously, not one by one.

In [11]:
views.sort_values(by=['hour', 'minute', 'second'], ascending=True)

,datetime,year,month,day,hour,minute,second,daytime
user,,,,,,,,
valentina,2020-05-15 00:00:13.222265,2020,5,15,0,0,13,night
valentina,2020-05-15 00:01:05.153738,2020,5,15,0,1,5,night
pavel,2020-05-12 00:01:27.764025,2020,5,12,0,1,27,night
pavel,2020-05-12 00:01:38.444917,2020,5,12,0,1,38,night
pavel,2020-05-12 00:01:55.395042,2020,5,12,0,1,55,night
...,...,...,...,...,...,...,...,...
artem,2020-04-29 23:48:14.208828,2020,4,29,23,48,14,evening
anatoliy,2020-05-09 23:53:55.599821,2020,5,9,23,53,55,evening
pavel,2020-05-09 23:54:54.260791,2020,5,9,23,54,54,evening


## Calculate the minimum and maximum for the hours and the mode for the daytime categories.

In [12]:
night_max = views[(views['daytime'] == 'night')]['hour'].max()
user_night_max = views[(views['hour'] == night_max)].index
print(f"{set(user_night_max)}, {night_max}")
morning_min = views[(views['daytime'] == 'morning')]['hour'].min()
user_morning_min = views[(views['hour'] == morning_min)].index
print(f"{set(user_morning_min)}, {morning_min}")
mode_for_hour = views['hour'].mode()
mode_for_daytime = views['daytime'].mode()
print(mode_for_daytime, mode_for_hour)

{'konstantin'}, 3
{'alexander'}, 8
0    evening
Name: daytime, dtype: category
Categories (6, object): ['night' < 'early morning' < 'morning' < 'afternoon' < 'early evening' < 'evening'] 0    22
Name: hour, dtype: int32


## Show the three earliest and latest hours of the day and their corresponding usernames using nsmallest() and nlargest().


In [17]:
smallest_hours = views.nsmallest(3, 'hour').index.tolist()
print(smallest_hours)
largest_hours = views.nlargest(3, 'hour').index.tolist()
print(largest_hours)

['artem', 'konstantin', 'konstantin']
['konstantin', 'artem', 'artem']


## Use the describe() method to get the basic statistics for the columns.

In [18]:
stats = views.describe()
iqr = stats.loc['75%', 'hour'] - stats.loc['25%', 'hour']
q1 = stats.loc['25%', 'hour']
q2 = stats.loc['75%', 'hour']
print(f"iqr = {iqr},  интервал: {q1} - {q2}")

iqr = 9.0,  интервал: 13.0 - 22.0
